# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"Dataset Name: {metadata['name']}\n")
print(f"Description: {metadata['description']}")
print(f"Date Published: {metadata['datePublished']}")
print(f"License: {metadata['license']}")

## 2. Data Overview
Review available record sets and fields. All references use entity `@id` values as required.

In [ ]:
# List all record sets (@ids) and their fields
record_sets = dataset.metadata.record_sets()
print("Record Sets found:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs['name']}")
    # List fields for the record set
    fields = rs.get('fields', [])
    for field in fields:
        print(f"    - Field @id: {field.get('@id')}, name: {field.get('name')}, type: {field.get('dataType')}")

### Example: Preview records from a record set
Here we display the first few records using their record set `@id`.

In [ ]:
# Pick the main survey record set @id (replace with the correct value from the dataset overview above)
# We'll use the first record set for demonstration
first_record_set = record_sets[0]['@id'] if record_sets else None

if first_record_set:
    print(f"First 3 records from record set {first_record_set}")
    for ix, rec in enumerate(dataset.records(record_set=first_record_set)):
        print(rec)
        if ix >= 2:
            break
else:
    print("No record sets available.")

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame for analysis. Reference all record sets by their `@id`s.

In [ ]:
# Extract data from all available record sets (@ids)
dataframes = {}

for rs in record_sets:
    record_set_id = rs['@id']
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nFields (columns) for record set {record_set_id}:")
    print(df.columns.tolist())
    print(f"Preview (first 3 rows):")
    print(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Let's apply typical preprocessing: filtering numeric records, normalizing, and grouping. We'll use columns referenced by their actual `@id` as required.

**Tip:** First, inspect available numeric and group fields by their `@id`.

In [ ]:
# Pick main record set for EDA
main_record_set_id = first_record_set
main_df = dataframes.get(main_record_set_id)

# Find a likely numeric field and group field from available columns
numeric_fields = [col for col in main_df.columns if 'score' in col or 'age' in col or 'GAD' in col or 'PHQ' in col or main_df[col].dtype in [int, float]]
group_fields = [col for col in main_df.columns if 'gender' in col or 'Gender' in col or 'education' in col or 'village' in col or main_df[col].dtype == object]

print(f"Numeric fields: {numeric_fields}")
print(f"Group fields: {group_fields}")

# Choose one numeric field for demonstration (replace with actual @id field if known)
if numeric_fields:
    numeric_field = numeric_fields[0]
    threshold = main_df[numeric_field].dropna().mean()
    filtered_df = main_df[main_df[numeric_field] > threshold]

    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if available
    if group_fields:
        group_field = group_fields[0]
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
else:
    print("No suitable numeric fields for EDA.")

## 5. Visualization
Visualize a numeric field distribution and its grouping.

All visualizations reference fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the numeric field
if numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouped data exists, barplot by group_field
    if group_fields and group_field in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"Mean {numeric_field} by {group_field} (@id)")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load, explore, and process the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`. The dataset offers demographic details and validated mental health assessment scores, referenced via Croissant `@id`s for reproducibility and clarity.

- We loaded and inspected metadata and available record sets.
- Extracted records and documented field `@id`s for each entity.
- Performed sample EDA, including numeric filtering, normalization, and grouping by categorical fields.
- Visualized data distributions by referencing the correct field `@id`.

Further exploration can use additional fields noted above (by their `@id`) and more advanced analyses relevant to mental health research.